<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Demo (AI): Kcal Snap — Snap a Food Photo, Get the Calories

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. **See that an LLM can *see*** — hand it an image, get back words
2. Turn a photo into a **structured calorie estimate** (Pydantic + `parse()`)
3. Wrap the model in a helper that reads an **uploaded** photo
4. Ship a **real web app** with Gradio — a public link you open on your phone
5. Add an **honesty check** — a confidence level and an "estimate, not medical advice" note

---

## 1. Environment Setup

Run these first. You'll need an **OpenAI API key** — the model that will *read* our photos. (Gemini's `gemini-2.0-flash` can also see images; we'll keep this demo to one provider.)

In [ ]:
# Install what we need: OpenAI (the vision model), Gradio (the web app), Pillow (images)
!pip install -q openai gradio pillow

In [ ]:
# Imports
import os
import base64
from io import BytesIO
from getpass import getpass
from openai import OpenAI
from pydantic import BaseModel, Field

In [ ]:
# API key (typed securely - not shown on screen)
openai_api_key = getpass("Enter your OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = openai_api_key

# A small, cheap model that can SEE images (proven in Colab, latest 2026)
OPENAI_MODEL = "gpt-4o-mini"

client = OpenAI()  # reads OPENAI_API_KEY from the environment
print("Ready. Vision model:", OPENAI_MODEL)

## 2. Can an LLM *see*?

So far you've asked models to work with **text**. Here's the surprise: the *same* model can take an **image** as input. You hand it a picture, it hands back words.

The trick is the message `content`: instead of a plain string, it's a **list** that can mix `text` and an `image_url`. Let's point it at a food photo and just ask *"what is this?"*

In [ ]:
# A public food photo. Change this URL to ANY image and re-run.
image_url = "https://upload.wikimedia.org/wikipedia/commons/a/a3/Eq_it-na_pizza-margherita_sep2005_sml.jpg"

# Notice: content is a LIST - some text + an image. That's the whole "vision" trick.
resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "What food is in this photo? Name it in one line."},
            {"type": "image_url", "image_url": {"url": image_url}},
        ],
    }],
)
print(resp.choices[0].message.content)

## 3. From "what is it" → calories

A sentence is nice, but an **app** needs data it can trust: a number for calories, numbers for protein / carbs / fat. So we ask for a **guaranteed shape** — a Pydantic model — and call `parse()`. Now the model must fill in *our* fields, and only those.

Notice the two honesty fields: `confidence` and `disclaimer`. A good AI app is upfront that it's only estimating.

In [ ]:
# The EXACT shape we want back. The model must fill these fields - it can't wander off.
class FoodItem(BaseModel):
    name: str
    calories: int

class CalorieEstimate(BaseModel):
    dish: str
    items: list[FoodItem]                       # each food on the plate
    total_calories: int
    protein_g: int
    carbs_g: int
    fat_g: int
    confidence: str = Field(description="low, medium, or high")
    disclaimer: str = Field(description="a short reminder that this is only an estimate")

# Same photo - change image_url and re-run to try another dish.
image_url = "https://upload.wikimedia.org/wikipedia/commons/a/a3/Eq_it-na_pizza-margherita_sep2005_sml.jpg"

completion = client.chat.completions.parse(
    model=OPENAI_MODEL,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Estimate the nutrition for the food in this image. Fill every field."},
            {"type": "image_url", "image_url": {"url": image_url}},
        ],
    }],
    response_format=CalorieEstimate,
)

food = completion.choices[0].message.parsed     # a real CalorieEstimate object
print(food.dish, "->", food.total_calories, "kcal")
print(f"protein {food.protein_g}g | carbs {food.carbs_g}g | fat {food.fat_g}g")
print("confidence:", food.confidence)

## 4. Wrap it: a photo goes in, a card comes out

In the app the photo won't be a public URL — it'll be a file the user just uploaded. A local image has no `url`, so we encode its bytes as **base64** and send that as a `data:` URL.

We fold everything into one function, `analyze_food(image)`, that returns a tidy Markdown card. This function *is* the brain of our app.

In [ ]:
# Turn an uploaded photo (a PIL image) into a Markdown nutrition card.
def analyze_food(image):
    if image is None:
        return "Please upload a food photo first."

    # 1) Encode the image as base64 (a local file has no URL, so we send its bytes)
    buffer = BytesIO()
    image.convert("RGB").save(buffer, format="JPEG")
    b64 = base64.b64encode(buffer.getvalue()).decode()
    data_url = f"data:image/jpeg;base64,{b64}"

    # 2) Ask for our guaranteed shape (same CalorieEstimate as section 3)
    completion = client.chat.completions.parse(
        model=OPENAI_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": "Estimate the nutrition for the food in this image. Fill every field."},
                {"type": "image_url", "image_url": {"url": data_url}},
            ],
        }],
        response_format=CalorieEstimate,
    )
    f = completion.choices[0].message.parsed

    # 3) Format the result as a friendly Markdown card
    items = "\n".join(f"- {it.name}: {it.calories} kcal" for it in f.items)
    return f"""## 🍽️ {f.dish}
# {f.total_calories} kcal
**Protein** {f.protein_g}g  ·  **Carbs** {f.carbs_g}g  ·  **Fat** {f.fat_g}g

{items}

_Confidence: {f.confidence}. {f.disclaimer}_"""

print("analyze_food is ready")

## 5. 🚀 The real app

Now the payoff. **Gradio** turns our function into a web page: an image upload on the left, the nutrition card on the right. In Colab, `launch()` prints a public **`gradio.live`** link — open it on your phone and point your camera at lunch.

A ten-line brain, wrapped in a few lines of UI. *That* is the shape of a real AI product.

In [ ]:
import gradio as gr

# inputs = a photo the user uploads; outputs = the Markdown card our function returns
app = gr.Interface(
    fn=analyze_food,
    inputs=gr.Image(type="pil", label="Upload a food photo"),
    outputs=gr.Markdown(),
    title="🍽️ Kcal Snap",
    description="Upload a photo of your food and get an instant calorie & macro estimate.",
)

# share=True gives a public link (auto-on in Colab). Open it on your phone!
app.launch(share=True)

## 6. Exercises

Fill in the blanks (`___`) and re-run. Small changes, big difference.

### Q1: Suggest a healthier swap

Add a new field so every result also suggests a lighter alternative (e.g. "grilled paneer instead of fried").

**Hints:** add one line to the model; the type that holds text is `str`. Then re-run the **app cell** (section 5).

In [ ]:
# Redefine CalorieEstimate with one extra field, then re-run the app cell.
class CalorieEstimate(BaseModel):
    dish: str
    items: list[FoodItem]
    total_calories: int
    protein_g: int
    carbs_g: int
    fat_g: int
    confidence: str
    disclaimer: str
    healthier_swap: ___ = Field(description="one lighter alternative to this dish")   # which TYPE holds text?

print("Field added - now re-run section 5 and try a photo.")

### Q2: Veg or non-veg?

Indian plates often need this. Add a boolean field the model fills in.

**Hints:** the type for true/false is `bool`.

In [ ]:
# Fill the blank so the model tells us if the dish is vegetarian.
class Dish(BaseModel):
    name: str
    is_vegetarian: ___          # true or false - which type?

completion = client.chat.completions.parse(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "Describe a plate of paneer butter masala."}],
    response_format=Dish,
)
d = completion.choices[0].message.parsed
print(d.name, "| vegetarian:", d.is_vegetarian)

### Q3: Make the app yours

Give the app your own title, and switch the public link **on**.

**Hints:** `share=` takes `True` or `False`; the title is just text.

In [ ]:
# Fill the blanks: your own title, and turn the public phone link on.
my_app = gr.Interface(
    fn=analyze_food,
    inputs=gr.Image(type="pil", label="Upload a food photo"),
    outputs=gr.Markdown(),
    title="___",                 # your app's name
)

my_app.launch(share=___)         # True = public phone link

---

### ✅ Recap

- An LLM can take an **image** as input — the message `content` becomes a list of `text` + `image_url`.
- A **Pydantic** model + `parse()` turns a fuzzy photo into a **guaranteed shape** your code can trust.
- An uploaded file has no URL, so we send its bytes as a **base64 `data:` URL**.
- **Gradio** wrapped a ten-line brain into a real, shareable web app.
- Good AI apps stay **honest** — we returned a `confidence` and an "only an estimate" note.

### 🔁 Make it yours (change ~3 lines)

The *same* app becomes:

- 🌿 **Plant-disease doctor** — photo of a leaf → disease + treatment (swap the prompt + fields)
- 🧾 **Receipt scanner** — photo of a bill → items, total, category
- 🐦 **"Shazam for nature"** — photo of a bird or insect → what it is + fun facts

The photo changes, the fields change — the **10-line pattern stays the same**.